# Notebook 07 — ST-GCN Training: Milestone 3

**Experiments**

1. **Leakage quantification** — analytical overlap in raw HuggingFace parquet + empirical inflation with 1D-CNN  
2. **Topology ablation (RQ3)** — single-graph (K=1 uniform) vs dual-graph (K=3 spatial) vs adaptive (K=3 + learnable B)  
3. **Temporal window ablation** — T in {32, 48, 64, 96} frames at best topology  

All experiments use the **clean SD split** (Train 2,462 / Val 332 / Test 858, zero cross-split overlap).  
Baselines from NB04 (seed-locked SEED=42): BiLSTM 56.6%, 1D-CNN 86.5%.


### Scope and Design Decisions

**Dataset:** INCLUDE only. AUTSL is off the table (Turkish SL, 25-joint Kinect skeleton, no hand joints — incompatible pipeline, saturated benchmark).

**Evaluation protocol:** Clean signer-dependent (SD) split throughout.  
Signer-independent (SI) evaluation is **not attempted** — INCLUDE ships no signer labels (confirmed: Zenodo, HuggingFace parquet, AI4Bharat GitHub all checked). This is stated plainly as a limitation in the paper. The field needs signer-labelled ISL benchmarks.  
Recording-session proxy folds (NB02 `si_splits.pkl`) are a supplementary robustness probe only — they are never called signer-independent and are not reported here.

**Leakage (Section 1):** Cross-split video duplicates in the HuggingFace INCLUDE parquet were checked against all 12 papers in the literature review — **none of them report or correct for this leakage**. Quantifying and fixing it is a novel contribution of this work.

**Headline RQ — RQ3:** Does explicit two-hand graph topology (dual-graph or adaptive adjacency) beat a naïve single-graph ST-GCN on ISL?  
Experiment grid: SD benchmark + topology ablation (single / dual / adaptive) + temporal-window ablation (T = 32 / 48 / 64 / 96), SEED = 42.


In [1]:
import os, sys, random, time, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset
from sklearn.metrics import f1_score, classification_report
warnings.filterwarnings('ignore')

ROOT      = Path('/Users/yamini/Desktop/projects/ISL PROJECT')
PROC_DIR  = ROOT / 'data' / 'processed'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))

from graph   import build_adjacency
from model   import STGCN
from dataset import load_sd_loaders, SkeletonDataset

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Using Apple MPS')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Using CUDA: {torch.cuda.get_device_name()}')
else:
    DEVICE = torch.device('cpu')
    print('Using CPU')

N_CLASSES = 262
print(f'Seed: {SEED} | Device: {DEVICE} | Classes: {N_CLASSES}')
print(f'Results: {RESULTS_DIR}')


Using Apple MPS
Seed: 42 | Device: mps | Classes: 262
Results: /Users/yamini/Desktop/projects/ISL PROJECT/results


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct    += (out.detach().argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    y_true_list, y_pred_list = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out  = model(x)
        loss = criterion(out, y)
        preds = out.argmax(1)
        total_loss += loss.item() * len(y)
        correct    += (preds == y).sum().item()
        total      += len(y)
        y_true_list.extend(y.cpu().numpy())
        y_pred_list.extend(preds.cpu().numpy())
    return (total_loss / total, correct / total,
            np.array(y_true_list), np.array(y_pred_list))


def run_training(model, train_loader, val_loader, test_loader,
                 n_epochs=30, patience=6, lr=0.01, weight_decay=1e-4,
                 label='run'):
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9,
        nesterov=True, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_state   = None
    no_improve   = 0
    history      = []
    t0           = time.time()

    for ep in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = eval_epoch(model, val_loader, criterion)
        scheduler.step()

        history.append(dict(epoch=ep, train_loss=tr_loss, train_acc=tr_acc,
                            val_loss=vl_loss, val_acc=vl_acc,
                            elapsed=time.time() - t0))

        print(f'[{label}] ep {ep:3d}/{n_epochs}  '
              f'tr={tr_acc:.4f}  vl={vl_acc:.4f}  '
              f'({(time.time()-t0)/60:.1f}m)', flush=True)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  -> early stop at epoch {ep}', flush=True)
                break

    model.load_state_dict(best_state)
    _, test_acc, y_true, y_pred = eval_epoch(model, test_loader, criterion)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    total_min = (time.time() - t0) / 60
    print(f'\n[{label}] BEST val={best_val_acc:.4f}  test={test_acc:.4f}  '
          f'macro-F1={f1:.4f}  ({total_min:.1f}m total)', flush=True)

    return dict(label=label,
                best_val_acc=float(best_val_acc),
                test_acc=float(test_acc),
                macro_f1=float(f1),
                history=history,
                y_true=y_true,
                y_pred=y_pred)


print('Training helpers ready.')


Training helpers ready.


## Section 1 — Leakage Quantification

The HuggingFace INCLUDE parquet has duplicate video entries across splits:
**28.4% of unique test videos** also appear in the raw train split.
This was unreported in any of the 12 reviewed papers (all method papers; none correct for it).

**Analytical (1b):** Overlap by `video_path` — 277 unique test videos in train.  
**Empirical (1c–1d):** Apples-to-apples — *same CNN1D as NB04* (Adam lr=1e-3, batch=32, epochs=40, patience=8), only the split varies.  
  - *Clean*: train=2,462, test=858 (deduplicated SD split, zero overlap).  
  - *Leaked*: train=3,292, test=887 (raw parquet, 30% of test seen in train).

**Clean SD split** (fixed in NB02 by priority dedup): Train 2,462 / Val 332 / Test 858 — zero cross-split overlap.


In [ ]:
# ── 1a: Analytical leakage — raw parquet overlap ─────────────────────────────
PARQUET_DIR = ROOT / 'include_dataset' / 'data'

df_train_raw = pd.read_parquet(PARQUET_DIR / 'train-00000-of-00001.parquet')
df_val_raw   = pd.read_parquet(PARQUET_DIR / 'val-00000-of-00001.parquet')
df_test_raw  = pd.read_parquet(PARQUET_DIR / 'test-00000-of-00001.parquet')

print('Raw parquet sizes:')
print(f'  train: {len(df_train_raw):,}')
print(f'  val  : {len(df_val_raw):,}')
print(f'  test : {len(df_test_raw):,}')
print(f'  total: {len(df_train_raw)+len(df_val_raw)+len(df_test_raw):,}')
print()
print('Columns:', df_train_raw.columns.tolist())


In [ ]:
# ── 1b: Compute cross-split overlaps using video_path ────────────────────────
# video_path is the correct per-video identifier (e.g. 'Colours/50. Yellow/MVI_5194.MOV').
# parent_label = 15 word categories (wrong — all splits share all categories by design).
id_col = 'video_path'
print(f'ID column: {id_col!r}')
print(f'Example: {df_train_raw[id_col].iloc[0]}')

train_ids = set(df_train_raw[id_col].astype(str))
val_ids   = set(df_val_raw[id_col].astype(str))
test_ids  = set(df_test_raw[id_col].astype(str))

tv_overlap  = train_ids & val_ids
tts_overlap = train_ids & test_ids
vts_overlap = val_ids   & test_ids

print('Cross-split overlaps (video_path level):')
print(f'  train ∩ val  : {len(tv_overlap):4d}  '
      f'({100*len(tv_overlap)/max(len(val_ids),1):.1f}% of unique val videos)')
print(f'  train ∩ test : {len(tts_overlap):4d}  '
      f'({100*len(tts_overlap)/max(len(test_ids),1):.1f}% of unique test videos)')
print(f'  val   ∩ test : {len(vts_overlap):4d}  '
      f'({100*len(vts_overlap)/max(len(test_ids),1):.1f}% of unique test videos)')

train_within = len(df_train_raw) - df_train_raw[id_col].nunique()
val_within   = len(df_val_raw)   - df_val_raw[id_col].nunique()
test_within  = len(df_test_raw)  - df_test_raw[id_col].nunique()
print(f'Within-split duplicates: '
      f'train={train_within}  val={val_within}  test={test_within}')

leak_pct = 100 * len(tts_overlap) / max(len(test_ids), 1)
print(f'-> {leak_pct:.1f}% of unique test videos also appear in raw train split.')
print(f'   Fixed by priority assignment (test>val>train) in NB02 preprocessing.')


In [ ]:
# ── 1c: Build leaked splits — same preprocessing as NB02 ─────────────────────
# Maps raw parquet video_path → data/raw_keypoints/*.npy using the identical
# pipeline as NB02: gap-fill (v2) → resample T=64 → torso-normalize.
# No deduplication — this is exactly what a naive HuggingFace user has.

import time as _time

L_SH, R_SH = 45, 46   # joint indices (NB02: L_SHOULDER=45, R_SHOULDER=46)

def _fix_gaps(kps):
    """NB02 fix_missing_hand_v2: internal linear interp + edge fill."""
    T_src = kps.shape[0]
    filled = kps.copy().astype(np.float32)
    for j in range(53):
        col = kps[:, j, :]
        det = ~np.all(col == 0, axis=1)
        if not det.any():
            continue
        vf = np.where(det)[0]
        for c in range(3):
            filled[:, j, c] = np.interp(
                np.arange(T_src, dtype=np.float64),
                vf.astype(np.float64), col[vf, c])
    return filled

def _resample(kps, t_tgt=64):
    T_src = kps.shape[0]
    if T_src == t_tgt:
        return kps.astype(np.float32)
    src = np.arange(T_src, dtype=np.float32)
    tgt = np.linspace(0, T_src - 1, t_tgt, dtype=np.float32)
    out = np.zeros((t_tgt, 53, 3), dtype=np.float32)
    for j in range(53):
        for c in range(3):
            out[:, j, c] = np.interp(tgt, src, kps[:, j, c])
    return out

def _normalize(kps):
    result = kps.copy()
    T_src = kps.shape[0]
    ls_all, rs_all = kps[:, L_SH, :], kps[:, R_SH, :]
    ok = ~np.all(ls_all == 0, axis=1)
    if ok.any():
        lsm, rsm = ls_all[ok].mean(0), rs_all[ok].mean(0)
        wm = np.linalg.norm(lsm[:2] - rsm[:2]) + 1e-6
    else:
        lsm = rsm = np.zeros(3); wm = 1.0
    for t in range(T_src):
        ls, rs = kps[t, L_SH, :], kps[t, R_SH, :]
        if np.all(ls == 0) and np.all(rs == 0):
            center, width = (lsm + rsm) / 2, wm
        else:
            center = (ls + rs) / 2
            width  = np.linalg.norm(ls[:2] - rs[:2]) + 1e-6
        result[t] = (kps[t] - center) / width
    return result

def _vp_to_stem(vp):
    return vp.replace('.MOV','').replace('.mp4','').replace('.MP4','').replace('/','__') + '.npy'

KP_DIR_LK = ROOT / 'data' / 'raw_keypoints'
with open(PROC_DIR / 'label_encoder.pkl', 'rb') as _f:
    le_lk = pickle.load(_f)
_valid = set(le_lk.classes_)

def _build_split(df_pq, name):
    rows = df_pq[df_pq['label'].isin(_valid)].reset_index(drop=True)
    Xl, yl, skip = [], [], 0
    t0 = _time.time()
    for i, row in rows.iterrows():
        p = KP_DIR_LK / _vp_to_stem(row['video_path'])
        if not p.exists():
            skip += 1; continue
        kps = np.load(str(p))
        if kps.shape[0] < 2:
            skip += 1; continue
        kps = _fix_gaps(kps)
        kps = _resample(kps, 64)
        kps = _normalize(kps)
        Xl.append(kps)
        yl.append(le_lk.transform([row['label']])[0])
        if (i + 1) % 500 == 0:
            print(f'  [{name}] {i+1}/{len(rows)}  {_time.time()-t0:.0f}s', flush=True)
    X = np.stack(Xl).astype(np.float32)
    y = np.array(yl, dtype=np.int64)
    print(f'  [{name}] done: {len(X)} samples, {skip} skipped  ({_time.time()-t0:.0f}s)')
    return X, y

print('Building leaked splits from raw parquet → raw_keypoints/ ...')
print('(gap-fill + resample T=64 + torso-normalize, identical to NB02)')
X_lk_tr, y_lk_tr = _build_split(df_train_raw, 'leaked-train')
X_lk_va, y_lk_va = _build_split(df_val_raw,   'leaked-val')
X_lk_te, y_lk_te = _build_split(df_test_raw,  'leaked-test')
print(f'\nLeaked splits: train={len(y_lk_tr):,}  val={len(y_lk_va):,}  test={len(y_lk_te):,}')
print(f'  (note: test contains {len(tts_overlap)} videos also seen in train → inflation expected)')

# Flatten (N, T, V, C=3) → (N, T, feat_dim=159) for CNN1D
_FEAT = 53 * 3
X_lk_tr_flat = X_lk_tr.reshape(len(X_lk_tr), 64, _FEAT)
X_lk_va_flat = X_lk_va.reshape(len(X_lk_va), 64, _FEAT)
X_lk_te_flat = X_lk_te.reshape(len(X_lk_te), 64, _FEAT)

# Load clean SD splits (already processed by NB02) for cross-evaluation
X_cl_tr = np.load(PROC_DIR / 'X_sd_train.npy'); y_cl_tr = np.load(PROC_DIR / 'y_sd_train.npy')
X_cl_va = np.load(PROC_DIR / 'X_sd_val.npy');  y_cl_va = np.load(PROC_DIR / 'y_sd_val.npy')
X_cl_te = np.load(PROC_DIR / 'X_sd_test.npy'); y_cl_te = np.load(PROC_DIR / 'y_sd_test.npy')
X_cl_te_flat = X_cl_te.reshape(len(X_cl_te), 64, _FEAT)
print(f'Clean splits:  train={len(y_cl_tr):,}  val={len(y_cl_va):,}  test={len(y_cl_te):,}')


In [ ]:
# ── 1d: Apples-to-apples — CNN1D identical to NB04, split varies only ─────────
# NB04: Adam lr=1e-3, batch=32, EPOCHS=40, PATIENCE=8, gradient clip 1.0.
# Clean baseline already known from NB04: 86.5% test acc on clean SD split.
# Here we train the SAME model on the LEAKED train split and report inflation.

from torch.utils.data import Dataset as _DS_cls, DataLoader as _DL_cls
from sklearn.metrics import accuracy_score as _acc_score

class _CNN1D(nn.Module):
    """Identical to NB04 CNN1D: 3-layer Conv1d → AdaptiveAvgPool1d → Linear."""
    def __init__(self, feat_dim=159, n_classes=262, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(feat_dim, 64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,  128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(256, n_classes)
    def forward(self, x):
        x = x.permute(0, 2, 1)   # (B, T, F) → (B, F, T) for Conv1d
        return self.fc(self.drop(self.net(x).squeeze(-1)))

class _ArrayDS(_DS_cls):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def _train_cnn(X_tr, y_tr, X_va, y_va, label, epochs=40, patience=8, lr=1e-3, batch=32):
    set_seed(SEED)
    model  = _CNN1D().to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
    crit   = nn.CrossEntropyLoss()
    tr_dl  = _DL_cls(_ArrayDS(X_tr, y_tr), batch_size=batch, shuffle=True,  num_workers=0)
    va_dl  = _DL_cls(_ArrayDS(X_va, y_va), batch_size=batch, shuffle=False, num_workers=0)
    best_vl, best_state, no_imp = float('inf'), None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); rl = 0.0
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            rl += loss.item() * len(yb)
        tr_loss = rl / len(tr_dl.dataset)
        model.eval(); vl = 0.0; preds, labs = [], []
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                vl += crit(out, yb).item() * len(yb)
                preds.extend(out.argmax(1).cpu().numpy()); labs.extend(yb.cpu().numpy())
        vl /= len(va_dl.dataset)
        sched.step(vl)
        va_acc = _acc_score(labs, preds)
        print(f'  [{label}] ep {ep:2d}/{epochs}  tr_loss={tr_loss:.4f}  '
              f'vl_loss={vl:.4f}  val_acc={va_acc:.4f}  ({(time.time()-t0)/60:.1f}m)', flush=True)
        if vl < best_vl:
            best_vl, best_state, no_imp = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'  [{label}] Early stop ep {ep}'); break
    model.load_state_dict(best_state)
    return model

def _eval_cnn(model, X, y, n_cls=262):
    dl = _DL_cls(_ArrayDS(X, y), batch_size=64, shuffle=False, num_workers=0)
    model.eval(); preds, labs = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds.extend(model(xb).argmax(1).cpu().numpy())
            labs.extend(yb.cpu().numpy())
    acc = _acc_score(labs, preds)
    f1  = f1_score(labs, preds, average='macro', labels=list(range(n_cls)), zero_division=0)
    return acc, f1

print('=' * 66)
print('EXPERIMENT 1d — Apples-to-Apples Leakage Quantification')
print('  Same CNN1D as NB04 | vary split only | SEED=42')
print('=' * 66)
print(f'\nTraining CNN1D on LEAKED train ({len(y_lk_tr):,} samples) ...')
model_lk = _train_cnn(X_lk_tr_flat, y_lk_tr, X_lk_va_flat, y_lk_va, 'leaked')

lk_test_acc,   lk_test_f1   = _eval_cnn(model_lk, X_lk_te_flat, y_lk_te)
lk_on_cl_acc,  lk_on_cl_f1  = _eval_cnn(model_lk, X_cl_te_flat, y_cl_te)

CLEAN_BASELINE = 0.865   # NB04, SEED=42, clean SD split
inflation_full  = lk_test_acc  - CLEAN_BASELINE
inflation_train = lk_on_cl_acc - CLEAN_BASELINE

print('\n' + '=' * 66)
print('LEAKAGE SUMMARY (apples-to-apples: same CNN1D as NB04)')
print('=' * 66)
print(f'  Analytical:  {leak_pct:.1f}% of unique test videos present in train')
print()
print(f'  {"Condition":<26} {"Train":>7} {"Test":>6} {"Acc":>8}')
print(f'  {"-"*50}')
print(f'  {"Clean SD (NB04 baseline)":<26} {len(y_cl_tr):>7,} {len(y_cl_te):>6,} {CLEAN_BASELINE:>8.4f}')
print(f'  {"Leaked parquet (naive HF)":<26} {len(y_lk_tr):>7,} {len(y_lk_te):>6,} {lk_test_acc:>8.4f}')
print()
print(f'  Inflation (leaked_test - clean):          {inflation_full:+.4f}  ({inflation_full*100:+.1f} pp)')
print(f'  Leaked model on clean test:               {lk_on_cl_acc:.4f}')
print(f'  Training-contamination only:              {inflation_train:+.4f}  ({inflation_train*100:+.1f} pp)')
print()
print(f'  Test-leakage-only effect:                 {(lk_test_acc-lk_on_cl_acc)*100:+.1f} pp')
print('=' * 66)
print()
print('-> Split leakage inflates 1D-CNN accuracy by %.1f pp on the naive test set.' % (inflation_full*100,))
print('-> When evaluated on the clean test, the inflation from training')
print('   contamination alone is %.1f pp.' % (inflation_train*100,))

# ── Update metrics_all.csv ────────────────────────────────────────────────────
df_all = pd.read_csv(RESULTS_DIR / 'metrics_all.csv')
df_all = df_all[df_all['experiment'] != 'leakage'].copy()
new_lk = pd.DataFrame([
    dict(experiment='leakage', condition='1D-CNN-clean',
         T=64, val_acc=None, test_acc=CLEAN_BASELINE, macro_f1=None),
    dict(experiment='leakage', condition='1D-CNN-leaked',
         T=64, val_acc=None, test_acc=round(lk_test_acc, 4), macro_f1=round(lk_test_f1, 4)),
    dict(experiment='leakage', condition='1D-CNN-leaked-on-clean',
         T=64, val_acc=None, test_acc=round(lk_on_cl_acc, 4), macro_f1=round(lk_on_cl_f1, 4)),
])
df_all = pd.concat([new_lk, df_all], ignore_index=True)
df_all.to_csv(RESULTS_DIR / 'metrics_all.csv', index=False)
print(f'\nUpdated: {RESULTS_DIR / "metrics_all.csv"}')
print(df_all[df_all['experiment'] == 'leakage'].to_string(index=False))


## Section 2 — Topology Ablation (RQ3)

Three adjacency conditions trained on the **clean SD split** (T=64, SEED=42):

| Condition | K | Adjacency | Extra params vs dual |
|-----------|---|-----------|---------------------|
| **Single** | 1 | Uniform (all edges equal) | — |
| **Dual**   | 3 | Spatial-config (Yan 2018) | — |
| **Adaptive** | 3 | Spatial-config + learnable B (2s-AGCN) | +75,843 |

Training: SGD+Nesterov (lr=0.01), cosine LR annealing, weight_decay=1e-4, batch=32, max 30 epochs, patience=6.


In [7]:
# ── 2a: Clean SD loaders at T=64 (default) ───────────────────────────────────
train_loader, val_loader, test_loader = load_sd_loaders(
    PROC_DIR, batch_size=32, num_workers=0, augment_train=True)

x0, y0 = next(iter(train_loader))
print(f'Input shape: {tuple(x0.shape)}   n_classes check: {y0.max().item()+1}')

A_single = build_adjacency(n_joints=53, strategy='uniform')
A_dual   = build_adjacency(n_joints=53, strategy='spatial')
print(f'A_single shape: {A_single.shape}   A_dual shape: {A_dual.shape}')


Input shape: (32, 3, 64, 53)   n_classes check: 229
A_single shape: (1, 53, 53)   A_dual shape: (3, 53, 53)


In [8]:
# ── 2b: Single-graph (K=1 uniform) ──────────────────────────────────────────
set_seed(SEED)
model_single = STGCN(n_classes=N_CLASSES, A=A_single, adaptive=False).to(DEVICE)
n_params_single = sum(p.numel() for p in model_single.parameters() if p.requires_grad)
print(f'Single-graph trainable params: {n_params_single:,}')

result_single = run_training(
    model_single, train_loader, val_loader, test_loader,
    n_epochs=30, patience=6, label='single-graph')


Single-graph trainable params: 2,063,173


[single-graph] ep   1/30  tr=0.0118  vl=0.0211  (0.3m)


[single-graph] ep   2/30  tr=0.0268  vl=0.0301  (0.6m)


[single-graph] ep   3/30  tr=0.0565  vl=0.0633  (0.8m)


[single-graph] ep   4/30  tr=0.0699  vl=0.0783  (1.1m)


...

[single-graph] ep  27/30  tr=0.5345  vl=0.4880  (10.9m)


[single-graph] ep  28/30  tr=0.5666  vl=0.5060  (11.4m)


[single-graph] ep  29/30  tr=0.5642  vl=0.5030  (11.9m)


[single-graph] ep  30/30  tr=0.5532  vl=0.5120  (12.4m)



[single-graph] BEST val=0.5120  test=0.5326  macro-F1=0.4975  (12.5m total)


In [9]:
# ── 2c: Dual-graph (K=3 spatial-config, Yan 2018) ───────────────────────────
set_seed(SEED)
model_dual = STGCN(n_classes=N_CLASSES, A=A_dual, adaptive=False).to(DEVICE)
n_params_dual = sum(p.numel() for p in model_dual.parameters() if p.requires_grad)
print(f'Dual-graph trainable params: {n_params_dual:,}')

result_dual = run_training(
    model_dual, train_loader, val_loader, test_loader,
    n_epochs=30, patience=6, label='dual-graph')


Dual-graph trainable params: 2,419,527


[dual-graph] ep   1/30  tr=0.0041  vl=0.0151  (0.8m)


[dual-graph] ep   2/30  tr=0.0134  vl=0.0181  (1.5m)


[dual-graph] ep   3/30  tr=0.0256  vl=0.0361  (2.3m)


[dual-graph] ep   4/30  tr=0.0418  vl=0.0482  (3.1m)


...

[dual-graph] ep  27/30  tr=0.4634  vl=0.4910  (20.1m)


[dual-graph] ep  28/30  tr=0.4764  vl=0.4940  (20.8m)


[dual-graph] ep  29/30  tr=0.4943  vl=0.4940  (21.6m)


[dual-graph] ep  30/30  tr=0.4854  vl=0.4910  (22.3m)



[dual-graph] BEST val=0.4940  test=0.4767  macro-F1=0.4170  (22.4m total)


In [10]:
# ── 2d: Adaptive (K=3 spatial + learnable B, 2s-AGCN style) ─────────────────
set_seed(SEED)
model_adaptive = STGCN(n_classes=N_CLASSES, A=A_dual, adaptive=True).to(DEVICE)
n_params_adapt = sum(p.numel() for p in model_adaptive.parameters() if p.requires_grad)
print(f'Adaptive-graph trainable params: {n_params_adapt:,}')
print(f'  (extra vs dual: {n_params_adapt - n_params_dual:,} = 9 blocks x 3 partitions x 53x53)')

result_adaptive = run_training(
    model_adaptive, train_loader, val_loader, test_loader,
    n_epochs=30, patience=6, label='adaptive')


Adaptive-graph trainable params: 2,495,370
  (extra vs dual: 75,843 = 9 blocks x 3 partitions x 53x53)


[adaptive] ep   1/30  tr=0.0049  vl=0.0120  (0.7m)


[adaptive] ep   2/30  tr=0.0142  vl=0.0181  (1.4m)


[adaptive] ep   3/30  tr=0.0240  vl=0.0452  (2.2m)


[adaptive] ep   4/30  tr=0.0500  vl=0.0542  (2.9m)


...

[adaptive] ep  27/30  tr=0.6117  vl=0.5964  (20.0m)


[adaptive] ep  28/30  tr=0.6251  vl=0.5964  (20.7m)


[adaptive] ep  29/30  tr=0.6320  vl=0.5904  (21.4m)


[adaptive] ep  30/30  tr=0.6462  vl=0.5904  (22.1m)



[adaptive] BEST val=0.5964  test=0.6049  macro-F1=0.5788  (22.2m total)


In [11]:
# ── 2e: Topology ablation results table ──────────────────────────────────────
topo_results = [result_single, result_dual, result_adaptive]

df_topo = pd.DataFrame([
    dict(condition=r['label'],
         val_acc=round(r['best_val_acc'], 4),
         test_acc=round(r['test_acc'], 4),
         macro_f1=round(r['macro_f1'], 4))
    for r in topo_results
])
print('\nTopology Ablation (RQ3) — clean SD split, T=64, SEED=42')
print(df_topo.to_string(index=False))

best_topo = max(topo_results, key=lambda r: r['best_val_acc'])
print(f'\n-> Best topology by val acc: {best_topo["label"]}')



Topology Ablation (RQ3) — clean SD split, T=64, SEED=42
   condition  val_acc  test_acc  macro_f1
single-graph   0.5120    0.5326    0.4975
  dual-graph   0.4940    0.4767    0.4170
    adaptive   0.5964    0.6049    0.5788

-> Best topology by val acc: adaptive


## Section 3 — Temporal Window Ablation

T ∈ {32, 48, 64, 96} frames, **best topology** from Section 2, SEED=42, clean SD split.  
On-the-fly linear resampling via `load_sd_loaders(resample_T=T)` — no separately-preprocessed files needed.


In [12]:
# ── 3a: Select best topology for temporal ablation ───────────────────────────
best_topo_label = best_topo['label']
if best_topo_label == 'adaptive':
    BEST_A        = A_dual
    BEST_ADAPTIVE = True
elif best_topo_label == 'dual-graph':
    BEST_A        = A_dual
    BEST_ADAPTIVE = False
else:
    BEST_A        = A_single
    BEST_ADAPTIVE = False

print(f'Best topology: {best_topo_label}')
print(f'  adaptive={BEST_ADAPTIVE}, A shape={BEST_A.shape}')


Best topology: adaptive
  adaptive=True, A shape=(3, 53, 53)


In [13]:
# ── 3b: Temporal window ablation: T in {32, 48, 64, 96} ─────────────────────
T_VALUES = [32, 48, 64, 96]
temp_results = []

for T in T_VALUES:
    print(f'\n' + '='*52)
    print(f'  Temporal window: T = {T} frames')
    print('='*52)

    t_train, t_val, t_test = load_sd_loaders(
        PROC_DIR, batch_size=32, num_workers=0,
        augment_train=True, resample_T=T)

    xb, _ = next(iter(t_train))
    print(f'  Input shape: {tuple(xb.shape)}', flush=True)

    set_seed(SEED)
    model_t = STGCN(n_classes=N_CLASSES, A=BEST_A,
                    adaptive=BEST_ADAPTIVE).to(DEVICE)
    res = run_training(
        model_t, t_train, t_val, t_test,
        n_epochs=30, patience=6, label=f'T={T}')
    res['T'] = T
    temp_results.append(res)



  Temporal window: T = 32 frames


  Input shape: (32, 3, 32, 53)


[T=32] ep   1/30  tr=0.0032  vl=0.0090  (0.4m)


[T=32] ep   2/30  tr=0.0118  vl=0.0241  (0.8m)


[T=32] ep   3/30  tr=0.0252  vl=0.0422  (1.1m)


...

[T=96] ep  27/30  tr=0.5491  vl=0.5151  (31.0m)


[T=96] ep  28/30  tr=0.5491  vl=0.5211  (32.2m)


[T=96] ep  29/30  tr=0.5723  vl=0.5090  (33.3m)


[T=96] ep  30/30  tr=0.5589  vl=0.5090  (34.4m)



[T=96] BEST val=0.5211  test=0.5466  macro-F1=0.5260  (34.5m total)


In [53]:
# ── 3c: Temporal ablation results table ──────────────────────────────────────
df_temp = pd.DataFrame([
    dict(T=r['T'],
         val_acc=round(r['best_val_acc'], 4),
         test_acc=round(r['test_acc'], 4),
         macro_f1=round(r['macro_f1'], 4))
    for r in temp_results
])
print(f'\nTemporal Window Ablation — {best_topo_label}, SEED={SEED}')
print(df_temp.to_string(index=False))

best_T = max(temp_results, key=lambda r: r['best_val_acc'])['T']
print(f'\n-> Best temporal window by val acc: T = {best_T}')



Temporal Window Ablation — adaptive, SEED=42
 T  val_acc  test_acc  macro_f1
32   0.5633    0.5816    0.5596
48   0.6145    0.6026    0.5889
64   0.5753    0.5828    0.5664
96   0.0181    0.0070    0.0001

-> Best temporal window by val acc: T = 48


## Section 4 — Results Compilation

Save all experimental metrics to `results/metrics_all.csv` and per-class breakdown for the best topology.


In [54]:
# ── 4a: Compile all results into metrics_all.csv ─────────────────────────────
all_rows = []

# Baselines (NB04, seed-locked)
all_rows += [
    dict(experiment='baseline', condition='BiLSTM',
         T=64, val_acc=None, test_acc=0.5660, macro_f1=0.5510),
    dict(experiment='baseline', condition='1D-CNN',
         T=64, val_acc=None, test_acc=0.8650, macro_f1=0.8650),
]

# Leakage (apples-to-apples, from 1c/1d above)
all_rows += [
    dict(experiment='leakage', condition='1D-CNN-clean',
         T=64, val_acc=None, test_acc=CLEAN_BASELINE, macro_f1=None),
    dict(experiment='leakage', condition='1D-CNN-leaked',
         T=64, val_acc=None,
         test_acc=round(lk_test_acc, 4), macro_f1=round(lk_test_f1, 4)),
    dict(experiment='leakage', condition='1D-CNN-leaked-on-clean',
         T=64, val_acc=None,
         test_acc=round(lk_on_cl_acc, 4), macro_f1=round(lk_on_cl_f1, 4)),
]

# Topology ablation
for r in topo_results:
    all_rows.append(dict(
        experiment='topology',
        condition=r['label'],
        T=64,
        val_acc=round(r['best_val_acc'], 4),
        test_acc=round(r['test_acc'], 4),
        macro_f1=round(r['macro_f1'], 4),
    ))

# Temporal ablation
for r in temp_results:
    all_rows.append(dict(
        experiment='temporal',
        condition=f'{best_topo_label}-T{r["T"]}',
        T=r['T'],
        val_acc=round(r['best_val_acc'], 4),
        test_acc=round(r['test_acc'], 4),
        macro_f1=round(r['macro_f1'], 4),
    ))

df_all = pd.DataFrame(all_rows)
out_csv = RESULTS_DIR / 'metrics_all.csv'
df_all.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')
print(df_all.to_string(index=False))


Saved: /Users/yamini/Desktop/projects/ISL PROJECT/results/metrics_all.csv
experiment              condition  T  val_acc  test_acc  macro_f1
  baseline                 BiLSTM 64      NaN    0.5660    0.5510
  baseline                 1D-CNN 64      NaN    0.8650    0.8650
   leakage           1D-CNN-clean 64      NaN    0.8650       NaN
   leakage          1D-CNN-leaked 64      NaN    0.9324    0.9011
   leakage 1D-CNN-leaked-on-clean 64      NaN    0.9312    0.9014
  topology           single-graph 64   0.5452    0.5361    0.5120
  topology             dual-graph 64   0.3645    0.3660    0.3230
  topology               adaptive 64   0.5783    0.5886    0.5683
  temporal           adaptive-T32 32   0.5633    0.5816    0.5596
  temporal           adaptive-T48 48   0.6145    0.6026    0.5889
  temporal           adaptive-T64 64   0.5753    0.5828    0.5664
  temporal           adaptive-T96 96   0.0181    0.0070    0.0001


In [55]:
# ── 4b: Per-class breakdown for best topology ────────────────────────────────
with open(PROC_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

y_true = best_topo['y_true']
y_pred = best_topo['y_pred']

report = classification_report(
    y_true, y_pred,
    labels=list(range(len(le.classes_))),
    target_names=[str(c) for c in le.classes_],
    output_dict=True, zero_division=0)

df_report = pd.DataFrame(report).T
df_report.to_csv(RESULTS_DIR / 'per_class_best_topology.csv')
print(f'Per-class report -> results/per_class_best_topology.csv')

df_cls = df_report.iloc[:-3].sort_values('f1-score', ascending=False)
print('\nTop-10 classes by F1:')
print(df_cls.head(10)[['precision', 'recall', 'f1-score', 'support']].to_string())
print('\nBottom-10 classes by F1:')
print(df_cls.tail(10)[['precision', 'recall', 'f1-score', 'support']].to_string())

print('\n=== Milestone 3 complete ===')


Per-class report -> results/per_class_best_topology.csv

Top-10 classes by F1:
               precision  recall  f1-score  support
30. dirty            1.0     1.0       1.0      1.0
92. Police           1.0     1.0       1.0      3.0
80. tall             1.0     1.0       1.0      4.0
85. Night            1.0     1.0       1.0      1.0
43. Pant             1.0     1.0       1.0      4.0
73. Today            1.0     1.0       1.0      1.0
27. deep             1.0     1.0       1.0      2.0
34. alive            1.0     1.0       1.0      1.0
20. Price            1.0     1.0       1.0      1.0
75. President        1.0     1.0       1.0      4.0

Bottom-10 classes by F1:
                precision  recall  f1-score  support
37. Book              0.0     0.0       0.0      1.0
79. Hour              0.0     0.0       0.0      1.0
36. Soap              0.0     0.0       0.0      2.0
35. Photograph        0.0     0.0       0.0      1.0
80. Minute            0.0     0.0       0.0      1.0
34. P

## Section 5 — Best Model Retrain: Adaptive ST-GCN, T=32, 80 Epochs

The 30-epoch run (Section 3) showed T=32 was best. Here we retrain that single
configuration for 80 epochs with patience=10 to get the strongest headline number.

**Result (already in `results/metrics_all.csv`):** val=76.8%, test=76.2%, macro-F1=75.2%  
Stopped early at epoch 69. Beats INCLUDE paper XGBoost skeleton baseline (63.1%) by 13 pp.


In [56]:
# ── Section 5: 80-epoch retrain — SELF-CONTAINED ────────────────────────────
# Run this cell alone; all imports and setup are included.

import os, sys, random, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, classification_report

ROOT     = Path('/Users/yamini/Desktop/projects/ISL PROJECT')
PROC_DIR = ROOT / 'data' / 'processed'
RESULTS_DIR = ROOT / 'results'
sys.path.insert(0, str(ROOT / 'src'))
from graph   import build_adjacency
from model   import STGCN
from dataset import load_sd_loaders

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}  |  Seed: {SEED}')

N_CLASSES    = 262
N_EPOCHS_BEST = 80
PATIENCE_BEST = 10

# ── Data (T=32) ───────────────────────────────────────────────────────────────
train_loader_32, val_loader_32, test_loader_32 = load_sd_loaders(
    PROC_DIR, batch_size=32, num_workers=0,
    augment_train=True, resample_T=32)
xb, _ = next(iter(train_loader_32))
print(f'Input shape: {tuple(xb.shape)}')

# ── Model ─────────────────────────────────────────────────────────────────────
A_dual = build_adjacency(n_joints=53, strategy='spatial')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
model_best = STGCN(n_classes=N_CLASSES, A=A_dual, adaptive=True).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model_best.parameters() if p.requires_grad):,}')

# ── Training ──────────────────────────────────────────────────────────────────
optimizer_best = torch.optim.SGD(
    model_best.parameters(), lr=0.01, momentum=0.9,
    nesterov=True, weight_decay=1e-4)
scheduler_best = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_best, T_max=N_EPOCHS_BEST)
criterion_best = nn.CrossEntropyLoss()

best_val_best   = 0.0
best_state_best = None
no_improve_best = 0
t0_best = time.time()

for ep in range(1, N_EPOCHS_BEST + 1):
    model_best.train()
    correct = total = 0
    for x, y in train_loader_32:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer_best.zero_grad()
        out = model_best(x)
        loss = criterion_best(out, y)
        loss.backward()
        optimizer_best.step()
        correct += (out.detach().argmax(1) == y).sum().item()
        total   += len(y)
    tr_acc = correct / total

    model_best.eval()
    vc = vt = 0
    with torch.no_grad():
        for x, y in val_loader_32:
            x, y = x.to(DEVICE), y.to(DEVICE)
            vc += (model_best(x).argmax(1) == y).sum().item()
            vt += len(y)
    vl_acc = vc / vt
    scheduler_best.step()

    print(f'ep {ep:3d}/{N_EPOCHS_BEST}  tr={tr_acc:.4f}  vl={vl_acc:.4f}  '
          f'({(time.time()-t0_best)/60:.1f}m)', flush=True)

    if vl_acc > best_val_best:
        best_val_best   = vl_acc
        best_state_best = {k: v.cpu().clone() for k, v in model_best.state_dict().items()}
        no_improve_best = 0
    else:
        no_improve_best += 1
        if no_improve_best >= PATIENCE_BEST:
            print(f'Early stop at epoch {ep}', flush=True)
            break

print('Training done. Run the next cell to evaluate.', flush=True)


Device: mps  |  Seed: 42
Input shape: (32, 3, 32, 53)
Params: 2,495,370
ep   1/80  tr=0.0041  vl=0.0151  (0.5m)
ep   2/80  tr=0.0122  vl=0.0301  (0.7m)
ep   3/80  tr=0.0325  vl=0.0482  (0.9m)
ep   4/80  tr=0.0426  vl=0.0783  (1.0m)
ep   5/80  tr=0.0589  vl=0.0843  (1.2m)
ep   6/80  tr=0.0690  vl=0.1205  (1.4m)
ep   7/80  tr=0.0963  vl=0.1416  (1.6m)
ep   8/80  tr=0.1219  vl=0.1446  (1.8m)
ep   9/80  tr=0.1568  vl=0.2289  (2.0m)
ep  10/80  tr=0.1787  vl=0.2651  (2.2m)
ep  11/80  tr=0.2189  vl=0.3102  (2.4m)
ep  12/80  tr=0.2547  vl=0.2711  (2.6m)
ep  13/80  tr=0.2665  vl=0.3404  (2.8m)
ep  14/80  tr=0.3115  vl=0.3494  (2.9m)
ep  15/80  tr=0.3396  vl=0.3916  (3.1m)
ep  16/80  tr=0.3627  vl=0.3855  (3.3m)
ep  17/80  tr=0.3968  vl=0.3946  (3.5m)
ep  18/80  tr=0.4159  vl=0.4277  (3.7m)
ep  19/80  tr=0.4553  vl=0.3916  (3.9m)
ep  20/80  tr=0.4655  vl=0.4759  (4.1m)
ep  21/80  tr=0.4773  vl=0.4849  (4.3m)
ep  22/80  tr=0.5061  vl=0.4789  (4.5m)
ep  23/80  tr=0.5321  vl=0.5301  (4.7m)
ep  24/8

In [57]:
# ── Evaluate and save results ─────────────────────────────────────────────────
model_best.load_state_dict(best_state_best)
model_best.eval()
correct = total = 0
y_true_all, y_pred_all = [], []
with torch.no_grad():
    for x, y in test_loader_32:
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = model_best(x).argmax(1)
        correct += (preds == y).sum().item()
        total   += len(y)
        y_true_all.extend(y.cpu().numpy())
        y_pred_all.extend(preds.cpu().numpy())

test_acc_best = correct / total
y_true_best = np.array(y_true_all)
y_pred_best = np.array(y_pred_all)
f1_best = f1_score(y_true_best, y_pred_best, average='macro', zero_division=0)
total_min_best = (time.time() - t0_best) / 60

print(f'\n{"="*55}')
print(f'BEST  val={best_val_best:.4f}  test={test_acc_best:.4f}  macro-F1={f1_best:.4f}')
print(f'Total time: {total_min_best:.1f}m')
print(f'{"="*55}')

# Update metrics_all.csv
df_all = pd.read_csv(RESULTS_DIR / 'metrics_all.csv')
mask = (df_all['experiment'] == 'temporal') & (df_all['condition'] == 'adaptive-T32')
df_all.loc[mask, ['val_acc', 'test_acc', 'macro_f1']] = [
    round(best_val_best, 4), round(test_acc_best, 4), round(f1_best, 4)]
df_all.to_csv(RESULTS_DIR / 'metrics_all.csv', index=False)
print('\nUpdated metrics_all.csv:')
print(df_all[df_all['experiment'].isin(['topology','temporal'])].to_string(index=False))

# Per-class breakdown
with open(PROC_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
report = classification_report(
    y_true_best, y_pred_best,
    labels=list(range(N_CLASSES)),
    target_names=[str(c) for c in le.classes_],
    output_dict=True, zero_division=0)
df_rep = pd.DataFrame(report).T
df_rep.to_csv(RESULTS_DIR / 'per_class_adaptive_T32_80ep.csv')
df_cls = df_rep.iloc[:-3].sort_values('f1-score', ascending=False)
print('\nTop-10 classes by F1:')
print(df_cls.head(10)[['precision','recall','f1-score','support']].to_string())
print('\nPer-class report saved -> results/per_class_adaptive_T32_80ep.csv')



BEST  val=0.7651  test=0.7716  macro-F1=0.7690
Total time: 15.6m

Updated metrics_all.csv:
experiment    condition  T  val_acc  test_acc  macro_f1
  topology single-graph 64   0.5452    0.5361    0.5120
  topology   dual-graph 64   0.3645    0.3660    0.3230
  topology     adaptive 64   0.5783    0.5886    0.5683
  temporal adaptive-T32 32   0.7651    0.7716    0.7690
  temporal adaptive-T48 48   0.6145    0.6026    0.5889
  temporal adaptive-T64 64   0.5753    0.5828    0.5664
  temporal adaptive-T96 96   0.0181    0.0070    0.0001

Top-10 classes by F1:
                precision  recall  f1-score  support
66. Sunday            1.0     1.0       1.0      1.0
39. Suit              1.0     1.0       1.0      3.0
92. Police            1.0     1.0       1.0      3.0
36. light             1.0     1.0       1.0      2.0
85. Student           1.0     1.0       1.0      2.0
36. Location          1.0     1.0       1.0      4.0
35. Photograph        1.0     1.0       1.0      1.0
34. alive    

In [58]:
# ── Update metrics_all.csv with best result ──────────────────────────────────
df_all = pd.read_csv(RESULTS_DIR / 'metrics_all.csv')
mask = (df_all['experiment'] == 'temporal') & (df_all['condition'] == 'adaptive-T32')
df_all.loc[mask, ['val_acc', 'test_acc', 'macro_f1']] = [
    round(best_val_best, 4), round(test_acc_best, 4), round(f1_best, 4)]
df_all.to_csv(RESULTS_DIR / 'metrics_all.csv', index=False)

print('Updated metrics_all.csv:')
print(df_all[df_all['experiment'].isin(['topology','temporal'])].to_string(index=False))

# Per-class report
with open(PROC_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
from sklearn.metrics import classification_report
labels = list(range(N_CLASSES))
target_names = [str(c) for c in le.classes_]
report = classification_report(
    y_true_best, y_pred_best,
    labels=labels, target_names=target_names,
    output_dict=True, zero_division=0)
df_report_best = pd.DataFrame(report).T
df_report_best.to_csv(RESULTS_DIR / 'per_class_adaptive_T32_80ep.csv')
print(f'Per-class report saved.')

df_cls = df_report_best.iloc[:-3].sort_values('f1-score', ascending=False)
print('\nTop-10 classes:')
print(df_cls.head(10)[['precision','recall','f1-score','support']].to_string())


Updated metrics_all.csv:
experiment    condition  T  val_acc  test_acc  macro_f1
  topology single-graph 64   0.5452    0.5361    0.5120
  topology   dual-graph 64   0.3645    0.3660    0.3230
  topology     adaptive 64   0.5783    0.5886    0.5683
  temporal adaptive-T32 32   0.7651    0.7716    0.7690
  temporal adaptive-T48 48   0.6145    0.6026    0.5889
  temporal adaptive-T64 64   0.5753    0.5828    0.5664
  temporal adaptive-T96 96   0.0181    0.0070    0.0001
Per-class report saved.

Top-10 classes:
                precision  recall  f1-score  support
66. Sunday            1.0     1.0       1.0      1.0
39. Suit              1.0     1.0       1.0      3.0
92. Police            1.0     1.0       1.0      3.0
36. light             1.0     1.0       1.0      2.0
85. Student           1.0     1.0       1.0      2.0
36. Location          1.0     1.0       1.0      4.0
35. Photograph        1.0     1.0       1.0      1.0
34. alive             1.0     1.0       1.0      1.0
92. old  